In [11]:
# CELL 1: Install Required Libraries
# We force upgrade sympy and torchao to resolve recent Colab version conflicts
!pip install -q --upgrade sympy torchao
!pip install -q transformers peft pandas Pillow torchvision tqdm
print("✅ Libraries installed successfully!")


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\Charlene C. Dilig\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


✅ Libraries installed successfully!



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\Charlene C. Dilig\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [12]:
# CELL 2: Mount Drive and Define Transforms
import torchvision.transforms as T
import torch

# 2. Define the Augmentation for TRAINING (Random Erasing)
train_transform = T.Compose([
    T.Resize((224, 224)),                    # Standardize size for FG-CLIP
    T.RandomHorizontalFlip(p=0.5),           # 50% chance to flip
    T.ColorJitter(brightness=0.2),           # Simulate variable lighting
    T.ToTensor(),                            # Convert to Tensor for Erasing
    T.RandomErasing(p=0.3, scale=(0.02, 0.15)), # 30% chance to mask a feature!
    T.ToPILImage()                           # Convert back for the HF Processor
])

# 3. Define the rule for VALIDATION (No Augmentation, pristine images only)
eval_transform = T.Compose([
    T.Resize((224, 224))
])

print("✅ Augmentations defined!")

✅ Augmentations defined!


In [13]:
# CELL 3: The PyTorch Dataset Class
import pandas as pd
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader

class LostAndFoundDataset(Dataset):
    def __init__(self, csv_file, image_dir, split_type, transform=None):
        full_data = pd.read_csv(csv_file)
        # Filter based on your 'train' or 'val' tags
        self.data = full_data[full_data['split'] == split_type].reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Open Lost Image and force 3-channel RGB (handles mixed PNG/JPGs safely)
        lost_img_path = os.path.join(self.image_dir, row['lost_image'])
        lost_image = Image.open(lost_img_path).convert('RGB')

        # Apply Random Erasing ONLY if this is the training set
        if self.transform:
            lost_image = self.transform(lost_image)

        # Return the Triplet
        return {
            'lost_image': lost_image,
            'positive_caption': row['positive_caption'],
            'hard_negative_caption': row['hard_negative_caption']
        }

# --- THE FIX: Custom Collate Function ---
# This stops PyTorch from crashing when it sees a PIL Image
def custom_collate(batch):
    return {
        'lost_image': [item['lost_image'] for item in batch],
        'positive_caption': [item['positive_caption'] for item in batch],
        'hard_negative_caption': [item['hard_negative_caption'] for item in batch]
    }

# UPDATE THESE PATHS IF YOUR FOLDERS ARE NAMED DIFFERENTLY IN DRIVE
CSV_PATH = '../captions/dataset_captionsV2.csv'
IMG_DIR = '../images'

# Initialize DataLoaders (Notice we added collate_fn=custom_collate here!)
train_dataset = LostAndFoundDataset(CSV_PATH, IMG_DIR, split_type='train', transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=custom_collate)

val_dataset = LostAndFoundDataset(CSV_PATH, IMG_DIR, split_type='val', transform=eval_transform)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=custom_collate)

print(f"✅ Loaded {len(train_dataset)} Training Pairs and {len(val_dataset)} Validation Pairs.")

✅ Loaded 420 Training Pairs and 90 Validation Pairs.


In [14]:
# CELL 4: Model Architecture & LoRA
from transformers import CLIPModel, CLIPProcessor
from peft import LoraConfig, get_peft_model

model_id = "qihoo360/fg-clip-base"
print(f"Downloading Base Model: {model_id}...")

processor = CLIPProcessor.from_pretrained(model_id)
base_model = CLIPModel.from_pretrained(model_id)

# Configure LoRA strictly for the Attention Layers (Query and Value)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none"
)

# Inject the adapters
lora_model = get_peft_model(base_model, lora_config)

# Move to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
lora_model.to(device)

print("\n✅ Model Ready! Printing trainable parameters for your paper:")
lora_model.print_trainable_parameters()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.



✅ Model Ready! Printing trainable parameters for your paper:
trainable params: 491,520 || all params: 150,112,257 || trainable%: 0.3274


In [15]:
print("============================================================")
print("🔎 PRE-TRAINING MODEL ARCHITECTURE INSPECTION")
print("============================================================")

try:
    # We reach directly into the PyTorch architecture of the loaded base_model
    # to find the convolutional layer responsible for chopping the image.
    patch_layer = base_model.vision_model.embeddings.patch_embedding

    patch_size = patch_layer.kernel_size[0]
    image_size = 224 # Standard CLIP input size
    num_patches = (image_size // patch_size) ** 2

    print(f"Target Layer: {patch_layer}")
    print(f"Kernel Size:  {patch_layer.kernel_size}  <-- This is your Patch Size!")
    print(f"Stride:       {patch_layer.stride}")

    print("\n--- Physical Grid Calculation ---")
    print(f"Math: {image_size} / {patch_size} = {image_size // patch_size}")
    print(f"Grid: {image_size // patch_size} x {image_size // patch_size} = {num_patches} total patches.")
    print(f"Tensor Shape: [1, {num_patches + 1}, 768] (Includes +1 for the CLS Token)")

    print("\n============================================================")
    print("🎯 FINAL VERDICT")
    print("============================================================")
    if patch_size == 16:
        print("✅ CONFIRMED: You are officially fine-tuning a ViT-B/16 model.")
    elif patch_size == 32:
        print("❌ WARNING: This is a ViT-B/32 model. Update your methodology!")
    else:
        print(f"⚠️ UNKNOWN: Patch size is {patch_size}.")

except AttributeError:
    print("⚠️ Error: Could not locate the patch_embedding layer. Ensure your variable is named 'base_model'.")

🔎 PRE-TRAINING MODEL ARCHITECTURE INSPECTION
Target Layer: Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
Kernel Size:  (16, 16)  <-- This is your Patch Size!
Stride:       (16, 16)

--- Physical Grid Calculation ---
Math: 224 / 16 = 14
Grid: 14 x 14 = 196 total patches.
Tensor Shape: [1, 197, 768] (Includes +1 for the CLS Token)

🎯 FINAL VERDICT
✅ CONFIRMED: You are officially fine-tuning a ViT-B/16 model.


In [24]:
# CELL 5: The Training Loop & Saving the Model
import torch.nn as nn
from torch.optim import AdamW
from tqdm import tqdm

# We ONLY pass the trainable LoRA parameters to the optimizer (Freezing the rest)
optimizer = AdamW(filter(lambda p: p.requires_grad, lora_model.parameters()), lr=5e-5)
loss_fn = nn.CrossEntropyLoss()

num_epochs = 15  # Adjust based on when validation loss stops improving

for epoch in range(num_epochs):
    lora_model.train()
    total_train_loss = 0

    # --- TRAINING PHASE ---
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    for batch in train_bar:
        optimizer.zero_grad()

        images = batch['lost_image']
        pos_texts = batch['positive_caption']
        neg_texts = batch['hard_negative_caption']

        # Combine texts: All Positives first, then All Negatives
        all_texts = pos_texts + neg_texts

        # Tokenize and encode
        inputs = processor(text=all_texts, images=images, return_tensors="pt", padding=True).to(device)
        outputs = lora_model(**inputs)

        # InfoNCE Math: Calculate similarity scores (logits)
        logits_per_image = outputs.logits_per_image

        # The correct match is always the index of the positive text
        labels = torch.arange(len(images)).to(device)
        loss = loss_fn(logits_per_image, labels)

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        train_bar.set_postfix({'loss': f"{loss.item():.4f}"})

    avg_train_loss = total_train_loss / len(train_loader)

    # --- VALIDATION PHASE ---
    lora_model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            images = batch['lost_image']
            pos_texts = batch['positive_caption']
            neg_texts = batch['hard_negative_caption']

            inputs = processor(text=pos_texts + neg_texts, images=images, return_tensors="pt", padding=True).to(device)
            outputs = lora_model(**inputs)

            labels = torch.arange(len(images)).to(device)
            loss = loss_fn(outputs.logits_per_image, labels)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    print(f"\n=> Epoch {epoch+1} Summary | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}\n")

# --- SAVE THE FINE-TUNED MODEL TO GOOGLE DRIVE ---
save_path = "../lorafinetuned/fgclip-lora-finetunedV1-1200"
lora_model.save_pretrained(save_path)
print(f"🎉 Training Complete! LoRA adapters successfully saved to: {save_path}")

import os
import datetime
print("\n--- Verifying Saved Files ---")

for f in os.listdir(save_path):
    full = os.path.join(save_path, f)
    mod_time = os.path.getmtime(full)
    # CHANGED: Converted the Unix timestamp into a human-readable date/time string
    readable_time = datetime.datetime.fromtimestamp(mod_time).strftime('%Y-%m-%d %H:%M:%S')
    print(f"📄 {f}: Saved at {readable_time}")

from peft import PeftModel
from transformers import CLIPModel
import json

print("\n--- Verifying Adapter Weights ---")
base = CLIPModel.from_pretrained("qihoo360/fg-clip-base")

loaded = PeftModel.from_pretrained(base, save_path)

with open(os.path.join(save_path, "adapter_config.json"), "r") as f:
    config = json.load(f)
print("✅ Adapter Config Loaded Successfully:")

print(json.dumps(config, indent=2))

Epoch 1/15 [Train]:   0%|          | 0/27 [00:00<?, ?it/s]

Epoch 1/15 [Train]: 100%|██████████| 27/27 [00:23<00:00,  1.16it/s, loss=0.3118]



=> Epoch 1 Summary | Train Loss: 0.4136 | Val Loss: 1.4889



Epoch 2/15 [Train]: 100%|██████████| 27/27 [00:23<00:00,  1.17it/s, loss=0.2459]



=> Epoch 2 Summary | Train Loss: 0.3921 | Val Loss: 1.5067



...


=> Epoch 13 Summary | Train Loss: 0.1539 | Val Loss: 1.5040



Epoch 14/15 [Train]: 100%|██████████| 27/27 [00:26<00:00,  1.00it/s, loss=0.1050]



=> Epoch 14 Summary | Train Loss: 0.1393 | Val Loss: 1.5126



Epoch 15/15 [Train]: 100%|██████████| 27/27 [00:26<00:00,  1.01it/s, loss=0.1278]



=> Epoch 15 Summary | Train Loss: 0.1411 | Val Loss: 1.5329

🎉 Training Complete! LoRA adapters successfully saved to: ../lorafinetuned/fgclip-lora-finetunedV1-1200

--- Verifying Saved Files ---
📄 adapter_config.json: Saved at 2026-05-04 13:46:32
📄 adapter_model.safetensors: Saved at 2026-05-04 13:46:32
📄 README.md: Saved at 2026-05-04 13:46:20

--- Verifying Adapter Weights ---
✅ Adapter Config Loaded Successfully:
{
  "alora_invocation_tokens": null,
  "alpha_pattern": {},
  "arrow_config": null,
  "auto_mapping": {
    "base_model_class": "CLIPModel",
    "parent_library": "transformers.models.clip.modeling_clip"
  },
  "base_model_name_or_path": "qihoo360/fg-clip-base",
  "bias": "none",
  "corda_config": null,
  "ensure_weight_tying": false,
  "eva_config": null,
  "exclude_modules": null,
  "fan_in_fan_out": false,
  "inference_mode": true,
  "init_lora_weights": true,
  "layer_replication": null,
  "layers_pattern": null,
  "layers_to_transform": null,
  "loftq_config": {},
  

In [23]:
# 📊 CELL: The Exhaustive "Final Boss" Experiment Summary
def print_final_boss_summary():
    import torch
    import transformers
    import peft
    import platform
    
    print("=" * 80)
    print("🔬 FINAL BOSS: EXHAUSTIVE EXPERIMENT CONFIGURATION & RESULTS SUMMARY")
    print("=" * 80)
    
    # 1. Hardware & Environment
    print("\n[1] HARDWARE & ENVIRONMENT")
    try: 
        print(f"    🖥️  OS:              {platform.system()} {platform.release()}")
        print(f"    🐍 Python Version:  {platform.python_version()}")
        print(f"    📦 PyTorch Version: {torch.__version__}")
        print(f"    📦 Transformers:    {transformers.__version__}")
        print(f"    📦 PEFT Version:    {peft.__version__}")
        if torch.cuda.is_available():
            print(f"    🚀 GPU:             {torch.cuda.get_device_name(0)}")
        else:
            print("    🐌 GPU:             None (CPU Training)")
        print(f"    🎲 Global Seed:     {torch.initial_seed()}")
    except Exception as e: print(f"    ⚠️  Error: {e}")

    # 2. Base Model & Dataset
    print("\n[2] MODEL & DATASET")
    try: print(f"    🤖 Base Model ID:       {model_id}")
    except NameError: pass
    try: print(f"    📏 Max Text Length:     {processor.tokenizer.model_max_length} tokens")
    except NameError: pass
    try:
        print(f"    🖼️  Train Images:        {len(train_dataset)}")
        print(f"    🖼️  Validation Images:   {len(val_dataset)}")
    except NameError: pass

    try:
        if 'hn_sampler' in globals():
            print(f"    ⛏️  Mining Strategy:     Hard Negative Category Mining")
        else:
            print(f"    ⛏️  Mining Strategy:     Standard Random Shuffle")
    except Exception: pass

    # 3. Model Architecture Details
    print("\n[3] ARCHITECTURE & LoRA")
    try:
        trainable, total = lora_model.get_nb_trainable_parameters()
        print(f"    📊 Trainable Params:    {trainable:,} ({(trainable/total)*100:.4f}% of total)")
        print(f"    🧠 LoRA Rank (r):       {lora_config.r}")
        print(f"    🧠 LoRA Alpha:          {lora_config.lora_alpha}")
        print(f"    🎯 Target Modules:      {lora_config.target_modules}")
        print(f"    💧 Dropout:             {lora_config.lora_dropout}")
    except NameError: pass

    # 4. Hyperparameters & Optimizer
    print("\n[4] HYPERPARAMETERS & TRAINING")
    try: 
        pg = optimizer.param_groups[0]
        print(f"    ⚙️  Optimizer:           {type(optimizer).__name__}")
        print(f"    📉 Learning Rate:       {pg['lr']}")
        print(f"    ⚓ Weight Decay:        {pg.get('weight_decay', 0.0)}")
        print(f"    🎢 Betas (B1, B2):      {pg.get('betas', 'N/A')}")
    except NameError: pass
    try: print(f"    ⚖️  Loss Function:       {type(loss_fn).__name__}")
    except NameError: pass
    try: print(f"    🌡️  Temperature:         {STRICT_TEMP}")
    except NameError: pass
    try: print(f"    📦 Batch Size:          {val_loader.batch_size}")
    except NameError: pass
    try: 
        print(f"    📈 Total Steps:         {total_steps}")
        print(f"    🔥 Warmup Steps:        {warmup_steps} (Cosine Scheduler)")
    except NameError: pass

    # 5. Training Results
    print("\n[5] TRAINING RESULTS")
    try: print(f"    🏆 Best Val Loss:       {best_val_loss:.4f}")
    except NameError: pass
    try: print(f"    🏁 Stopped at Epoch:    {epoch+1} / {num_epochs}")
    except NameError: pass
    try: print(f"    🛑 Early Stop Patience: {patience} epochs")
    except NameError: pass

    # 6. Output Details
    print("\n[6] OUTPUT")
    try: print(f"    💾 Saved To:            {save_path}")
    except NameError: pass

    # 7. Augmentations
    print("\n[7] DATA AUGMENTATIONS")
    try:
        print(train_transform)
    except NameError:
        print("    ⚠️  Could not find train_transform variable.")
        
    print("=" * 80)

# Run the function
print_final_boss_summary()


🔬 FINAL BOSS: EXHAUSTIVE EXPERIMENT CONFIGURATION & RESULTS SUMMARY

[1] HARDWARE & ENVIRONMENT
    🖥️  OS:              Windows 10
    🐍 Python Version:  3.13.3
    📦 PyTorch Version: 2.6.0+cu124
    📦 Transformers:    4.57.1
    📦 PEFT Version:    0.19.1
    🚀 GPU:             NVIDIA GeForce RTX 3050 Laptop GPU
    🎲 Global Seed:     1497608227700

[2] MODEL & DATASET
    🤖 Base Model ID:       qihoo360/fg-clip-base
    📏 Max Text Length:     77 tokens
    🖼️  Train Images:        420
    🖼️  Validation Images:   90
    ⛏️  Mining Strategy:     Standard Random Shuffle

[3] ARCHITECTURE & LoRA
    📊 Trainable Params:    491,520 (0.3274% of total)
    🧠 LoRA Rank (r):       8
    🧠 LoRA Alpha:          16
    🎯 Target Modules:      {'q_proj', 'v_proj'}
    💧 Dropout:             0.1

[4] HYPERPARAMETERS & TRAINING
    ⚙️  Optimizer:           AdamW
    📉 Learning Rate:       5e-05
    ⚓ Weight Decay:        0.01
    🎢 Betas (B1, B2):      (0.9, 0.999)
    ⚖️  Loss Function:       Cross